In [ ]:
from abc import ABC, abstractmethod
import pyspark.sql.functions as F

In [ ]:
class TaxiSilverTable(SilverTable, TableWithDate, ABC):

    DATE_COLUMN = "pickup_datetime"
    VALUE_COLUMNS = ["trip_distance", "fare_amount", "total_amount", "tip_amount"]
    @property
    def RESULT_COLUMNS(self):
        return ["pickup_datetime", "dropoff_datetime", "pickup_location_id", "dropoff_location_id"] + self.ADDITIONAL_RESULT_COLUMNS

    @property
    @abstractmethod
    def ADDITIONAL_RESULT_COLUMNS(self) -> list:
        pass
    
    @property
    @abstractmethod
    def TAXI_TYPE_ID(self) -> int:
        pass

    def enrich_by_taxi_type_id(self):
        new_df = self.df.withColumn("taxi_type_id", F.lit(self.TAXI_TYPE_ID).cast("smallint"))
        return type(self)(new_df)

    def enrich_by_value_columns(self):
        new_df = self.df
        for column in self.VALUE_COLUMNS:
            if column not in new_df.columns:
                new_df = new_df.withColumn(column, F.lit(None).cast("double"))
        return type(self)(new_df)

    def enrich_by_date(self):
        new_df = self.df.withColumn("date", F.to_date("pickup_datetime"))
        return type(self)(new_df)
